# 09. USDA NRCS SNOTEL Daily Telemetry Download & Zarr Consolidation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/flowracle/notebooks/data_preprocessing/09_download_snotel_daily.ipynb)

This notebook programmatically queries active USDA Natural Resources
Conservation Service (NRCS) **SNOTEL** station metadata via the public AWDB
REST API, downloads daily station time series (`WTEQ`, `PREC`, `TMAX`, `TMIN`,
`TAVG`, `PRCP`) from the NRCS Report Generator, converts imperial units to SI
(inches $\to$ meters, feet $\to$ meters, $^\circ\text{F} \to \text{K}$,
longitude in $[0, 360)^\circ\text{E}$), and consolidates them into the Zarr
store consumed by Flowracle:
- **Output path**: `${FLOWRACLE_DATA_DIR}/snotel-zarr/snotel.zarr`

> **Runtime Tip**: Setting `FILTER_CA_DOMAIN_ONLY = True` (default)
> downloads only the ~85 active SNOTEL stations inside Flowracle's extended
> California snow-fed domain (`32.5–42.0°N`, `124.0–115.0°W`),
> completing in ~2 minutes. Set `FILTER_CA_DOMAIN_ONLY = False` to mirror
> the entire ~900-station Western US SNOTEL network.

##### Copyright 2026 The Google Research Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

---
*Disclaimer: This is not an officially supported Google product.*

In [ ]:
!pip install -q "numpy>=1.26.0" "pandas>=2.0.0" "requests>=2.31.0" "beautifulsoup4>=4.12.0" "lxml>=4.9.0" "xarray>=2023.12.0" "zarr>=2.16.0,<3.0.0" "dask>=2023.12.0" "tqdm>=4.66.0"

In [ ]:
import io
import os
import shutil
import time

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm
import xarray as xr

FLOWRACLE_DATA_DIR = os.environ.get("FLOWRACLE_DATA_DIR", "./flowracle_data")
OUTPUT_DIR = os.path.join(FLOWRACLE_DATA_DIR, "snotel-zarr")
RAW_CSV_DIR = os.path.join(OUTPUT_DIR, "raw_csv")
os.makedirs(RAW_CSV_DIR, exist_ok=True)

FILTER_CA_DOMAIN_ONLY = True
CA_DOMAIN_LAT_MIN, CA_DOMAIN_LAT_MAX = 32.5, 42.0
CA_DOMAIN_LON_MIN, CA_DOMAIN_LON_MAX = -124.0, -115.0
START_DATE = "1979-01-01"

AWDB_STATIONS_URL = (
    "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/stations"
    "?stationTriplets=*:*:SNTL&activeOnly=true"
)
NRCS_REPORT_URL = (
    "https://wcc.sc.egov.usda.gov/reportGenerator/view_csv/"
    "customSingleStationReport/daily/{station_id}:{state}:SNTL%7Cid=%22%22%7Cname/"
    "POR_BEGIN,POR_END/WTEQ::value,PREC::value,TMAX::value,TMIN::value,TAVG::value,PRCP::value"
)

## 1. Fetch Active SNOTEL Station Inventory from USDA NRCS AWDB

In [ ]:
resp = requests.get(AWDB_STATIONS_URL, timeout=60)
resp.raise_for_status()
stations_json = resp.json()

records = []
for item in stations_json:
  triplet = item.get("stationTriplet", "")
  parts = triplet.split(":")
  if len(parts) != 3 or parts[2] != "SNTL":
    continue
  try:
    stn_id = int(parts[0])
  except ValueError:
    continue
  lat = item.get("latitude")
  lon = item.get("longitude")
  elev_ft = item.get("elevation")
  if lat is None or lon is None:
    continue
  records.append({
      "station_id": stn_id,
      "state": parts[1],
      "name": item.get("name", ""),
      "latitude": float(lat),
      "longitude": float(lon),
      "elevation_m": float(elev_ft) * 0.3048 if elev_ft is not None else np.nan,
  })

stations_df = (
    pd.DataFrame(records)
    .drop_duplicates(subset=["station_id"])
    .sort_values("station_id")
)
if FILTER_CA_DOMAIN_ONLY:
  mask = (
      (stations_df["latitude"] >= CA_DOMAIN_LAT_MIN)
      & (stations_df["latitude"] <= CA_DOMAIN_LAT_MAX)
      & (stations_df["longitude"] >= CA_DOMAIN_LON_MIN)
      & (stations_df["longitude"] <= CA_DOMAIN_LON_MAX)
  )
  stations_df = stations_df[mask].reset_index(drop=True)

stations_csv = os.path.join(OUTPUT_DIR, "snotel_stations_metadata.csv")
stations_df.to_csv(stations_csv, index=False)
print(
    f"Selected {len(stations_df)} active SNOTEL stations (saved metadata to"
    f" {stations_csv})."
)

## 2. Download Daily Station Reports from NRCS Report Generator

In [ ]:
for _, row in tqdm(
    stations_df.iterrows(), total=len(stations_df), desc="Downloading SNOTEL"
):
  stn_id, state = int(row["station_id"]), row["state"]
  out_csv = os.path.join(RAW_CSV_DIR, f"snotel_{state}_{stn_id}.csv")
  if os.path.exists(out_csv) and os.path.getsize(out_csv) > 0:
    continue
  url = NRCS_REPORT_URL.format(station_id=stn_id, state=state)
  for attempt in range(4):
    try:
      r = requests.get(url, timeout=30)
      r.raise_for_status()
      df_stn = pd.read_csv(io.StringIO(r.text), comment="#")
      if not df_stn.empty and "Date" in df_stn.columns:
        df_stn.to_csv(out_csv, index=False)
      break
    except requests.RequestException:
      time.sleep(2**attempt)

## 3. Convert to SI Units & Consolidate into `snotel.zarr`

In [ ]:
IN_TO_M = 0.0254


def fahrenheit_to_kelvin(arr):
  return (arr - 32.0) * (5.0 / 9.0) + 273.15


col_map = {
    "Snow Water Equivalent (in) Start of Day Values": (
        "snow_water_equivalent_start_of_day_PST",
        "in",
    ),
    "Precipitation Accumulation (in) Start of Day Values": (
        "precipitation_accumulation_start_of_day_PST",
        "in",
    ),
    "Air Temperature Maximum (degF)": ("air_temperature_daily_max_PST", "degF"),
    "Air Temperature Minimum (degF)": ("air_temperature_daily_min_PST", "degF"),
    "Air Temperature Average (degF)": ("air_temperature_daily_avg_PST", "degF"),
    "Precipitation Increment (in)": ("precipitation_increment_daily_PST", "in"),
}

per_var_frames = {target_name: {} for target_name, _ in col_map.values()}
valid_rows = []

for _, row in stations_df.iterrows():
  stn_id, state = int(row["station_id"]), row["state"]
  csv_path = os.path.join(RAW_CSV_DIR, f"snotel_{state}_{stn_id}.csv")
  if not os.path.exists(csv_path):
    continue
  df = pd.read_csv(csv_path)
  if "Date" not in df.columns:
    continue
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
  df = df.dropna(subset=["Date"])
  df = df[df["Date"] >= pd.Timestamp(START_DATE)].set_index("Date").sort_index()
  df = df[~df.index.duplicated(keep="last")]
  if df.empty:
    continue
  valid_rows.append(row)
  for raw_col, (target_name, unit_kind) in col_map.items():
    if raw_col in df.columns:
      vals = pd.to_numeric(df[raw_col], errors="coerce").astype(np.float32)
      vals = vals * IN_TO_M if unit_kind == "in" else fahrenheit_to_kelvin(vals)
      per_var_frames[target_name][stn_id] = vals

meta_valid = pd.DataFrame(valid_rows).set_index("station_id")
stn_index = meta_valid.index.values.astype(np.int64)
full_time = pd.date_range(
    start=START_DATE,
    end=max(s.index.max() for d in per_var_frames.values() for s in d.values()),
    freq="D",
)

data_vars = {}
for target_name, stn_dict in per_var_frames.items():
  mat = np.full((len(full_time), len(stn_index)), np.nan, dtype=np.float32)
  for j, sid in enumerate(stn_index):
    if sid in stn_dict:
      aligned_s = stn_dict[sid].reindex(full_time)
      mat[:, j] = aligned_s.values.astype(np.float32)
  units = "Kelvin" if "temperature" in target_name else "meters"
  data_vars[target_name] = (("time", "index"), mat, {"units": units})

# Daily SWE increment
swe_mat = data_vars["snow_water_equivalent_start_of_day_PST"][1]
swe_diff = np.diff(swe_mat, axis=0, prepend=np.nan).astype(np.float32)
data_vars["snow_water_equivalent_daily_diff_PST"] = (
    ("time", "index"),
    swe_diff,
    {"units": "meters"},
)
data_vars["latitude"] = (
    ("index",),
    meta_valid["latitude"].values.astype(np.float32),
    {},
)
data_vars["longitude"] = (
    ("index",),
    ((meta_valid["longitude"].values + 360.0) % 360.0).astype(np.float32),
    {},
)
data_vars["elevation"] = (
    ("index",),
    meta_valid["elevation_m"].values.astype(np.float32),
    {"units": "meters"},
)
data_vars["time_PST"] = (("time",), full_time.values, {})

snotel_ds = xr.Dataset(
    data_vars=data_vars,
    coords={"time": full_time.values, "index": stn_index},
    attrs={
        "description": (
            "Daily measurements from the USDA NRCS SNOTEL network in PST,"
            " converted to SI units (meters, Kelvin, longitude in [0, 360)"
            " deg E)."
        )
    },
)

snotel_zarr_path = os.path.join(OUTPUT_DIR, "snotel.zarr")
if os.path.exists(snotel_zarr_path):
  shutil.rmtree(snotel_zarr_path)
snotel_ds.to_zarr(snotel_zarr_path, mode="w")
print(f"Wrote Zarr store: {snotel_zarr_path}")
print(snotel_ds)